<a href="https://colab.research.google.com/github/HudaSaffo/fashion-recommendation-system/blob/week3-segmentation/image_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!pip -q install opencv-python-headless pillow pandas tqdm matplotlib

  Preparing metadata (setup.py) ... done


 Setup, Google Drive Connection, and Catalog Loading

In [2]:
import hashlib
import json
import shutil
import urllib.request
from urllib.error import HTTPError
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

INPUT_DIR = Path('/content/drive/MyDrive/polyvore_image_cache')
CATALOG_PATH = Path('/content/drive/MyDrive/catalog.parquet')
OUTPUT_DIR = Path('/content/drive/MyDrive/polyvore_segmented_cache')
MASK_DIR = OUTPUT_DIR / 'masks'
CUTOUT_DIR = OUTPUT_DIR / 'cutouts'
META_DIR = OUTPUT_DIR / 'metadata'
REVIEW_DIR = OUTPUT_DIR / 'review'
for directory in (MASK_DIR, CUTOUT_DIR, META_DIR, REVIEW_DIR):
    directory.mkdir(parents=True, exist_ok=True)

assert INPUT_DIR.exists(), f'Week 1 image cache not found: {INPUT_DIR}'
assert CATALOG_PATH.exists(), f'Week 2 catalog not found: {CATALOG_PATH}'

catalog = pd.read_parquet(CATALOG_PATH).copy()
required_columns = {'item_id', 'category', 'color'}
missing_columns = required_columns - set(catalog.columns)
assert not missing_columns, f'catalog.parquet is missing: {sorted(missing_columns)}'
catalog['item_id'] = catalog['item_id'].astype(str).str.strip()
catalog = catalog[catalog['item_id'].ne('')].drop_duplicates('item_id', keep='last').copy()
CATALOG_BY_ITEM_ID = catalog.set_index('item_id').to_dict('index')

MAX_IMAGES = None
OVERWRITE = False
CUTOUT_PADDING_PX = 8
PIPELINE_VERSION = 'week3_v3_catalog_safe_colors_padded_cutouts'

# Only these exact color values may use the fast white-background route.
CONFIDENT_NONWHITE_COLORS = {
    'black', 'blue', 'red', 'green', 'yellow', 'pink', 'gold', 'silver',
    'grey', 'gray', 'brown', 'beige',
}

print(f'Catalog records: {len(CATALOG_BY_ITEM_ID):,}')
print(f'Image cache:     {INPUT_DIR}')
print(f'Output folder:   {OUTPUT_DIR}')
print('Smoke test is ON (MAX_IMAGES = 20). Inspect results before processing all images.') # this is before, now full batch is on

Catalog records: 3,762
Image cache:     /content/drive/MyDrive/polyvore_image_cache
Output folder:   /content/drive/MyDrive/polyvore_segmented_cache
Smoke test is ON (MAX_IMAGES = 20). Inspect results before processing all images.


 Download and Initialize the Segment Anything Model (SAM)

In [3]:
CHECKPOINT = Path('/content/sam_vit_b_01ec64.pth')
CHECKPOINT_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

def download_sam_checkpoint(url, destination):
    partial = destination.with_suffix(destination.suffix + '.part')
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        with urllib.request.urlopen(request, timeout=120) as response, open(partial, 'wb') as output:
            shutil.copyfileobj(response, output)
        partial.replace(destination)
    except HTTPError as error:
        partial.unlink(missing_ok=True)
        raise RuntimeError(
            f'Checkpoint download was denied ({error.code}). Retry this cell once; if it persists, '
            f'download {url} in your browser and upload it to {destination}.'
        ) from error

if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size < 100_000_000:
    CHECKPOINT.unlink(missing_ok=True)
    print('Downloading the official SAM ViT-B checkpoint')
    download_sam_checkpoint(CHECKPOINT_URL, CHECKPOINT)


import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
sam = sam_model_registry['vit_b'](checkpoint=str(CHECKPOINT)).to(device=DEVICE)
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.92,
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=250,
)
print(f'SAM ViT-B ready on {DEVICE}.')

SAM ViT-B ready on cuda.


Segmentation Helper Functions and Safe Colour Routing

In [4]:
def json_safe(value):
    # Convert pandas/numpy values without changing the source catalog
    if isinstance(value, (list, tuple, dict)):
        return json.loads(json.dumps(value, default=str))
    if value is None or pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value

def safe_catalog_metadata(item_id):
    return {key: json_safe(value) for key, value in CATALOG_BY_ITEM_ID[item_id].items()}

def white_background_mask(rgb, threshold=245):
    return np.all(rgb >= threshold, axis=2)

def has_clean_white_background(rgb):
    white = white_background_mask(rgb)
    h, w = white.shape
    border_width = max(2, min(h, w) // 30)
    border = np.concatenate([
        white[:border_width].ravel(), white[-border_width:].ravel(),
        white[:, :border_width].ravel(), white[:, -border_width:].ravel(),
    ])
    return bool(border.mean() >= 0.97 and white.mean() >= 0.55)

def color_route(item_metadata):
    # Return the reason for the safe segmentation route without modifying catalog color
    raw_color = item_metadata.get('color')
    color = '' if raw_color is None or pd.isna(raw_color) else str(raw_color).strip().lower()
    if color in CONFIDENT_NONWHITE_COLORS:
        return 'shortcut_eligible_nonwhite'
    if not color:
        return 'sam_missing_color'
    if color == 'multi/neutral':
        return 'sam_multi_neutral_color'
    if color in {'white', 'off-white', 'off white', 'ivory', 'cream', 'ecru'}:
        return 'sam_white_like_color'
    return 'sam_uncertain_color'

def should_use_white_background_shortcut(rgb, item_metadata):
    # White, multi/neutral, missing, and uncertain colors always go through SAM.
    return color_route(item_metadata) == 'shortcut_eligible_nonwhite' and has_clean_white_background(rgb)

def largest_center_component(mask):
    count, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    if count <= 1:
        return mask.astype(bool)
    h, w = mask.shape
    best_label, best_score = 0, -np.inf
    for label in range(1, count):
        area = stats[label, cv2.CC_STAT_AREA]
        if area < h * w * 0.002:
            continue
        cx, cy = centroids[label]
        distance = np.hypot(cx - w / 2, cy - h / 2) / np.hypot(w / 2, h / 2)
        score = area / (h * w) - 0.20 * distance
        if score > best_score:
            best_label, best_score = label, score
    return labels == best_label if best_label else mask.astype(bool)

def foreground_from_white(rgb):
    mask = ~white_background_mask(rgb)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=1)
    return largest_center_component(mask)

def choose_sam_mask(rgb, candidates):
    h, w = rgb.shape[:2]
    best_mask, best_score = None, -np.inf
    for candidate in candidates:
        mask = candidate['segmentation']
        area_ratio = float(mask.mean())
        if not 0.005 <= area_ratio <= 0.92:
            continue
        x, y, bw, bh = candidate['bbox']
        distance = np.hypot(x + bw / 2 - w / 2, y + bh / 2 - h / 2) / np.hypot(w / 2, h / 2)
        touches = int(x <= 1) + int(y <= 1) + int(x + bw >= w - 1) + int(y + bh >= h - 1)
        score = (2 * candidate['predicted_iou'] + candidate['stability_score']
                 + 0.20 * area_ratio - 0.65 * distance - 0.20 * touches)
        if score > best_score:
            best_mask, best_score = mask, score
    return largest_center_component(best_mask) if best_mask is not None else None

def bbox_from_mask(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1

def padded_bbox(bbox, image_width, image_height, padding):
    x1, y1, x2, y2 = bbox
    return (max(0, x1 - padding), max(0, y1 - padding),
            min(image_width, x2 + padding), min(image_height, y2 + padding))

def save_outputs(rgb, mask, item_id, source_path, method, item_metadata):
    original_bbox = bbox_from_mask(mask)
    if original_bbox is None:
        return None
    height, width = mask.shape
    crop_bbox = padded_bbox(original_bbox, width, height, CUTOUT_PADDING_PX)
    x1, y1, x2, y2 = original_bbox
    cx1, cy1, cx2, cy2 = crop_bbox
    rgba = np.dstack([rgb, (mask * 255).astype(np.uint8)])

    Image.fromarray((mask * 255).astype(np.uint8), mode='L').save(MASK_DIR / f'{item_id}.png')
    Image.fromarray(rgba[cy1:cy2, cx1:cx2], mode='RGBA').save(CUTOUT_DIR / f'{item_id}.png')

    metadata = safe_catalog_metadata(item_id)
    flat_catalog_metadata = {f'catalog_{key}': value for key, value in metadata.items()}
    record = {
        'pipeline_version': PIPELINE_VERSION,
        'item_id': item_id,
        'source_path': str(source_path),
        'mask_path': str(MASK_DIR / f'{item_id}.png'),
        'cutout_path': str(CUTOUT_DIR / f'{item_id}.png'),
        'method': method,
        'color_route': color_route(item_metadata),
        'category': metadata.get('category'),
        'color': metadata.get('color'),
        'original_mask_bbox_xyxy': [x1, y1, x2, y2],
        'cutout_crop_bbox_xyxy': [cx1, cy1, cx2, cy2],
        'padding_px': CUTOUT_PADDING_PX,
        'original_bbox_width': x2 - x1,
        'original_bbox_height': y2 - y1,
        'aspect_ratio': round((x2 - x1) / (y2 - y1), 4),
        'mask_area_ratio': round(float(mask.mean()), 4),
        'catalog_metadata': metadata,
        **flat_catalog_metadata,
    }
    with open(META_DIR / f'{item_id}.json', 'w', encoding='utf-8') as stream:
        json.dump(record, stream, indent=2, ensure_ascii=False, default=str)
    return record

Batch Segmentation and Transparent Cutout Generation

In [5]:
image_files = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg'}
)
if MAX_IMAGES is not None:
    image_files = image_files[:MAX_IMAGES]
print(f'Processing {len(image_files):,} image files (MAX_IMAGES={MAX_IMAGES}).')

run_records, failures, skipped_not_in_catalog = [], [], []
for source_path in tqdm(image_files, desc='Segmenting'):
    item_id = source_path.stem
    if item_id not in CATALOG_BY_ITEM_ID:
        skipped_not_in_catalog.append({
            'item_id': item_id, 'source_path': str(source_path),
            'reason': 'item_id_not_found_in_catalog',
        })
        continue

    metadata_path = META_DIR / f'{item_id}.json'
    if not OVERWRITE and (CUTOUT_DIR / f'{item_id}.png').exists() and metadata_path.exists():
        with open(metadata_path, encoding='utf-8') as stream:
            existing = json.load(stream)
        if existing.get('pipeline_version') == PIPELINE_VERSION:
            continue

    try:
        rgb = np.asarray(Image.open(source_path).convert('RGB'))
        if min(rgb.shape[:2]) < 16:
            raise ValueError('image is too small')
        item_metadata = CATALOG_BY_ITEM_ID[item_id]
        if should_use_white_background_shortcut(rgb, item_metadata):
            mask, method = foreground_from_white(rgb), 'white_background_shortcut'
        else:
            mask, method = choose_sam_mask(rgb, mask_generator.generate(rgb)), 'sam_vit_b'
        if mask is None or not mask.any():
            raise ValueError('no usable foreground mask')
        record = save_outputs(rgb, mask, item_id, source_path, method, item_metadata)
        if record is None:
            raise ValueError('empty mask bounding box')
        run_records.append(record)
    except Exception as exc:
        failures.append({'item_id': item_id, 'source_path': str(source_path), 'error': str(exc)})

pd.DataFrame(run_records).to_csv(OUTPUT_DIR / 'segmentation_manifest_latest.csv', index=False)
pd.DataFrame(failures).to_csv(OUTPUT_DIR / 'segmentation_failures_latest.csv', index=False)
pd.DataFrame(skipped_not_in_catalog).to_csv(OUTPUT_DIR / 'segmentation_skipped_not_in_catalog_latest.csv', index=False)
print(f'Saved: {len(run_records):,}; failed: {len(failures):,}; skipped (not catalogued): {len(skipped_not_in_catalog):,}.')

Processing 3,762 image files (MAX_IMAGES=None).


Segmenting:   0%|          | 0/3762 [00:00<?, ?it/s]

/tmp/ipykernel_2699/3977874824.py:105: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray((mask * 255).astype(np.uint8), mode='L').save(MASK_DIR / f'{item_id}.png')
/tmp/ipykernel_2699/3977874824.py:106: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba[cy1:cy2, cx1:cx2], mode='RGBA').save(CUTOUT_DIR / f'{item_id}.png')


Saved: 3,554; failed: 9; skipped (not catalogued): 0.


Build the Complete Segmentation Manifest

In [6]:
all_records = []
for metadata_path in META_DIR.glob('*.json'):
    with open(metadata_path, encoding='utf-8') as stream:
        record = json.load(stream)
    if record.get('pipeline_version') == PIPELINE_VERSION:
        all_records.append(record)
manifest = pd.DataFrame(all_records)
if len(manifest):
    manifest = manifest.sort_values('item_id').reset_index(drop=True)
manifest.to_csv(OUTPUT_DIR / 'segmentation_manifest.csv', index=False)
print(f'Complete manifest: {len(manifest):,} successful cutouts')
display(manifest[['item_id', 'category', 'color', 'method', 'color_route', 'aspect_ratio']].head())

Complete manifest: 3,753 successful cutouts


,item_id,category,color,method,color_route,aspect_ratio
0,100263578,bottoms,multi/neutral,sam_vit_b,sam_multi_neutral_color,0.9627
1,100758952,bottoms,grey,sam_vit_b,shortcut_eligible_nonwhite,0.3445
2,100798103,dresses,multi/neutral,sam_vit_b,sam_multi_neutral_color,1.0325
3,101003390,shoes,white,sam_vit_b,sam_white_like_color,0.3402
4,101124203,accessories,multi/neutral,sam_vit_b,sam_multi_neutral_color,0.4538


Create the 100-Image Mask Quality Review Page

In [7]:
REVIEW_SAMPLE_SIZE = 100
if manifest.empty:
    raise RuntimeError('No cutouts available. Run the batch segmentation cell first.')

sam_risk_routes = {'sam_white_like_color', 'sam_multi_neutral_color', 'sam_missing_color', 'sam_uncertain_color'}
risky = manifest[manifest['color_route'].isin(sam_risk_routes)].copy()
risky = risky.sample(n=min(50, len(risky)), random_state=42)
remaining = manifest.drop(index=risky.index)
random_count = min(REVIEW_SAMPLE_SIZE - len(risky), len(remaining))
review = pd.concat([risky, remaining.sample(n=random_count, random_state=42)], ignore_index=True)
review = review.sample(frac=1, random_state=42).reset_index(drop=True)
review['score'] = ''
review['review_notes'] = ''
review.to_csv(REVIEW_DIR / 'review_scores.csv', index=False)

thumb_dir = REVIEW_DIR / 'thumbnails'
thumb_dir.mkdir(exist_ok=True)
cards = []
for index, row in review.iterrows():
    image = Image.open(row.cutout_path).convert('RGBA')
    preview = Image.new('RGBA', image.size, (235, 235, 235, 255))
    preview.alpha_composite(image)
    preview.thumbnail((220, 220))
    thumb_name = f'{index:03d}.png'
    preview.convert('RGB').save(thumb_dir / thumb_name)
    risk_label = 'COLOR-SAFETY REVIEW' if row.color_route in sam_risk_routes else 'representative sample'
    cards.append(f'''<article><img src="thumbnails/{thumb_name}"><p><b>{index:03d}</b> - {row.item_id}</p><p>Category: {row.category}; color: {row.color}</p><p>Method: {row.method}; route: {row.color_route}</p><p><b>{risk_label}</b></p><p>Score: ☐ 0 failed &nbsp; ☐ 1 rough &nbsp; ☐ 2 clean</p></article>''')

html = '''<!doctype html><html><head><meta charset="utf-8"><title>Week 3 mask quality review</title><style>body{font-family:Arial;margin:24px;background:#fafafa}.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(245px,1fr));gap:14px}article{background:white;padding:10px;border:1px solid #ddd;border-radius:8px}img{width:100%;height:220px;object-fit:contain;background:#eee}p{font-size:12px;margin:6px 0}</style></head><body><h1>Week 3 Mask Quality Review</h1><p>Score each item in <code>review_scores.csv</code>: 0 = failed, 1 = usable but rough, 2 = clean enough for collage. Target: at least 80% score 1 or 2. This sample intentionally includes up to 50 white/uncertain-color SAM cases.</p><div class="grid">''' + ''.join(cards) + '</div></body></html>'
(REVIEW_DIR / 'index.html').write_text(html, encoding='utf-8')
print(f'Review page: {REVIEW_DIR / "index.html"}')
print(f'Scoring sheet: {REVIEW_DIR / "review_scores.csv"}')

Review page: /content/drive/MyDrive/polyvore_segmented_cache/review/index.html
Scoring sheet: /content/drive/MyDrive/polyvore_segmented_cache/review/review_scores.csv


Calculate Cutout Quality Evaluation Metrics

In [4]:
scores = pd.read_csv(REVIEW_DIR / 'review_scores.csv')
scored = pd.to_numeric(scores['score'], errors='coerce').dropna()
assert scored.isin([0, 1, 2]).all()
if scored.empty:
    print('No scores entered yet.')
else:
    usable_rate = (scored >= 1).mean()
    clean_rate = (scored == 2).mean()
    print(f'Reviewed: {len(scored)}/{len(scores)}')
    print(f'Usable cutouts: {usable_rate:.1%} (target: at least 80.0%)')
    print(f'Clean for collage: {clean_rate:.1%}')
    display(scores['score'].value_counts(dropna=False).sort_index())

Reviewed: 100/100
Usable cutouts: 82.0% (target: at least 80.0%)
Clean for collage: 64.0%


,count
score,
0,18
1,18
2,64
